# Native Multimodal Fusion: Gemma 4 E4B

Hateful meme detection with **joint fusion**, one transformer reads image + caption together.
This version LoRA fine-tunes Gemma on the **train** split so it is compared fairly against the
late-fusion model. `P(hateful)` comes from the Yes/No logits, so AUROC is computable.

- **train** - LoRA fine-tuning
- **val** - sanity / model-selection signal
- **test** - the reported number

In [ ]:
!pip install "transformers>=5.10.2" accelerate bitsandbytes peft
!pip install scikit-learn tqdm

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import torch
import numpy as np
import bitsandbytes as bnb
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, roc_curve
from google.colab import drive, userdata, files

In [ ]:
login(token=userdata.get("HF_TOKEN"))
print("Logged into HuggingFace")

## Configuration

In [ ]:
MODEL_NAME = "google/gemma-4-E4B-it"
DRIVE_BASE = "/content/drive/MyDrive/IIS Project/Data"
DEVICE     = "cuda"

DATA_DIR = f"{DRIVE_BASE}/100-samples"
# DATA_DIR = f"{DRIVE_BASE}/full-samples"

QUESTION = ("This is a meme. Look at the image and the caption together. "
            "Is it hateful? Answer with a single word: Yes or No.")


EPOCHS     = 3
LR         = 2e-4
LORA_R     = 16
LORA_ALPHA = 32
GRAD_ACCUM = 8
SEED       = 42

torch.manual_seed(SEED)

## Data

Three disjoint splits from the frozen set on Drive. Each split has `<split>.jsonl` beside a
`<split>/images/` folder, and every row's `img` field is `images/<id>.png`.

In [ ]:
drive.mount('/content/drive')

def load_split(split):
    with open(f"{DATA_DIR}/{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_image(split, row):
    return Image.open(f"{DATA_DIR}/{split}/{row['img']}").convert("RGB")

train_rows = load_split("train")
val_rows   = load_split("val")
test_rows  = load_split("test")

for name, rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
    print(f"{name:5s}: {len(rows):4d} memes | hateful {sum(r['label'] for r in rows)}")

## Model

Load Gemma in 4-bit and attach a LoRA adapter to the language-model projections.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

PROJ = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")

def is_proj(name):
    parts = name.split(".")
    return parts[-1] in PROJ or (len(parts) >= 2 and parts[-2] in PROJ)

lin4 = [n for n, m in model.named_modules() if isinstance(m, bnb.nn.Linear4bit) and is_proj(n)]
lm = [n for n in lin4 if any(k in n for k in ("language_model", "text_model"))]
target_modules = lm or [
    n for n in lin4
    if not any(t in n for t in ("vision", "audio", "multi_modal", "mm_", "siglip", "mobilenet", "timm"))
]

print(f"proj Linear4bit found: {len(lin4)} | LoRA targets: {len(target_modules)}")
print("examples:", target_modules[:3] if target_modules else "NONE")
if not target_modules:
    print("TOP-LEVEL:", [n for n, _ in model.named_children()])
    print("all proj Linear4bit:", lin4[:10])

lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

model = get_peft_model(model, lora)
model.print_trainable_parameters()
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Scoring

Yes/No token ids (case + leading-space variants), then one forward pass per meme reading the logits at the first answer position. `P(hateful) = softmax([no, yes])[1]`.

In [ ]:
def token_ids(words):
    ids = {processor.tokenizer.encode(w, add_special_tokens=False)[0] for w in words}
    return sorted(ids)

YES_IDS = token_ids(["Yes", " Yes", "yes", " yes"])
NO_IDS  = token_ids(["No", " No", "no", " no"])

@torch.no_grad()
def hateful_prob(image, caption):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text": f'Caption: "{caption}"\n{QUESTION}'},
    ]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(DEVICE)

    logits = model(**inputs).logits[0, -1, :].float()
    yes = torch.logsumexp(logits[YES_IDS], dim=0)
    no  = torch.logsumexp(logits[NO_IDS],  dim=0)
    return torch.softmax(torch.stack([no, yes]), dim=0)[1].item()

def score(rows, split, desc):
    model.eval()
    probs, labels = [], []
    for row in tqdm(rows, desc=desc):
        probs.append(hateful_prob(load_image(split, row), row["text"]))
        labels.append(int(row["label"]))
    return probs, labels

## Fine-tuning (LoRA)

Supervised fine-tuning on the train split. The target is the single Yes/No answer token, the loss is masked so only the answer contributes. After each epoch the model is scored on val, and the adapter from the best-val-AUROC epoch is kept for testing.

In [ ]:
ANSWER = {1: "Yes", 0: "No"}

def build_inputs(row, split):
    image = load_image(split, row)
    user = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text": f'Caption: "{row["text"]}"\n{QUESTION}'},
    ]}]
    prompt = processor.apply_chat_template(
        user, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    )
    full_msgs = user + [{"role": "assistant",
                         "content": [{"type": "text", "text": ANSWER[int(row["label"])]}]}]
    full = processor.apply_chat_template(
        full_msgs, tokenize=True, return_dict=True, return_tensors="pt",
    )
    labels = full["input_ids"].clone()
    labels[:, : prompt["input_ids"].shape[1]] = -100
    full["labels"] = labels
    return full

optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)

best_val, best_epoch, best_state, best_vp, best_vl = -1.0, 0, None, None, None
for epoch in range(EPOCHS):
    model.train()
    order = torch.randperm(len(train_rows),
                           generator=torch.Generator().manual_seed(SEED + epoch)).tolist()
    running = 0.0
    optim.zero_grad()
    for step, i in enumerate(tqdm(order, desc=f"epoch {epoch+1}/{EPOCHS}"), start=1):
        batch = build_inputs(train_rows[i], "train").to(DEVICE)
        loss = model(**batch).loss
        (loss / GRAD_ACCUM).backward()
        running += loss.item()
        if step % GRAD_ACCUM == 0 or step == len(order):
            optim.step()
            optim.zero_grad()
    vp, vl = score(val_rows, "val", f"val (epoch {epoch+1})")
    val_auc = roc_auc_score(vl, vp)
    print(f"epoch {epoch+1}: train loss {running/len(order):.4f} | val AUROC {val_auc:.3f}")
    if val_auc > best_val:
        best_val, best_epoch = val_auc, epoch + 1
        best_vp, best_vl = vp, vl
        best_state = {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}

set_peft_model_state_dict(model, best_state)
print(f"Selected epoch {best_epoch} | val AUROC {best_val:.3f}")

## Results

In [ ]:
def pick_threshold(labels, probs):
    ts = np.linspace(0.05, 0.95, 19)
    return float(max(ts, key=lambda t: f1_score(labels, [int(p >= t) for p in probs])))

def bootstrap_auc_ci(labels, probs, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    y, s = np.array(labels), np.array(probs)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    vals = []
    for _ in range(n):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        vals.append(roc_auc_score(y[idx], s[idx]))
    return np.percentile(vals, [2.5, 97.5])

threshold = pick_threshold(best_vl, best_vp)

probs, labels = score(test_rows, "test", "test")
preds = [int(p >= threshold) for p in probs]
auc = roc_auc_score(labels, probs)
lo, hi = bootstrap_auc_ci(labels, probs)

results = pd.DataFrame([{
    "model":        "Gemma 4 E4B (LoRA, joint fusion)",
    "AUROC":        round(auc, 4),
    "AUROC 95% CI": f"[{lo:.3f}, {hi:.3f}]",
    "threshold":    round(threshold, 2),
    "Accuracy":     round(accuracy_score(labels, preds), 4),
    "F1":           round(f1_score(labels, preds), 4),
    "n":            len(labels),
}])
results

In [ ]:
fpr, tpr, _ = roc_curve(labels, probs)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"Gemma 4 E4B LoRA (AUROC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC: Gemma (LoRA, joint fusion)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
pd.DataFrame({
    "id":           [r["id"] for r in test_rows],
    "label":        labels,
    "prob_hateful": probs,
    "pred":         preds,
}).to_csv("gemma4_predictions.csv", index=False)
files.download("gemma4_predictions.csv")